In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score)


In [7]:
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)


In [19]:
print(df.shape)

(7043, 22)


In [20]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["churn"] = (df["Churn"] == "Yes").astype(int)

num = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
cat = ["Contract", "InternetService", "PaymentMethod",
           "PaperlessBilling", "Partner", "Dependents", "TechSupport"]
X = pd.get_dummies(df[num + cat], columns=cat, dtype=int)
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=42, stratify=y_train
)

print(X.shape, X.isna().sum().sum(), round(y.mean(), 4))
print(X_train.shape, X_test.shape, X_tr.shape, X_val.shape)
print(y_val.sum(), y_test.sum())

(7043, 23) 11 0.2654
(5634, 23) (1409, 23) (4225, 23) (1409, 23)
374 374


- to ensure lock the random generation process, if not it will randomly reshuffle to a completely different mix.
- Because if we fill on thhe whole table before splitting, a fixed seed means the same shuffle every run. The main reason here is fairness. Day 9’s logistic regression and today’s forest must be judged on exactly the same validation customers, or you’d be comparing two models on two different exams.

In [24]:
def report(name, y_true, p, t=0.5):
    pred = (p >= t)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    print(f"{name} t={t}: tn={tn} fp={fp} fn={fn} tp={tp} | "
          f"acc = {accuracy_score(y_true, pred):.4f} "
          f"prec = {precision_score(y_true, pred, zero_division=0):.4f} "
          f"rec = {recall_score(y_true, pred):.4f} "
          f"f1 = {f1_score(y_true, pred):.4f}")

In [25]:
base = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
print("Baseline val acc", round(accuracy_score(y_val, base.predict(X_val)), 4))

Baseline val acc 0.7346


In [26]:
lr = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
]).fit(X_tr, y_tr)
p_lr = lr.predict_proba(X_val)[:, 1]
report("LR val", y_val, p_lr)
print("LR val auc", round(roc_auc_score(y_val, p_lr), 4),
      "train acc", round(lr.score(X_tr, y_tr), 4))


LR val t=0.5: tn=938 fp=97 fn=171 tp=203 | acc = 0.8098 prec = 0.6767 rec = 0.5428 f1 = 0.6024
LR val auc 0.8357 train acc 0.7974


- the baseline's 0.7346 tells me that the model baseline is quite high in a imbalance dataset. So a score of 0.8098 is quite okay where it just beat the baseline by 0.0752. 
- It means that if you pick a random churner and a random stayer, the model gives the churner the higher probability 83.57% of the time. 0.5 is random and 1.0 is perfect. 
- 0.8357 is the validation AUC, so that’s the one to beat.

In [27]:
for depth in (None, 3, 5):
    tree = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("model", DecisionTreeClassifier(max_depth=depth, random_state=42))
    ]).fit(X_tr, y_tr)
    t = tree.named_steps["model"]
    p = tree.predict_proba(X_val)[:, 1]
    print(depth, "| depth", t.get_depth(), "leaves", t.get_n_leaves(),
          "| train", round(tree.score(X_tr, y_tr), 4),
          "val", round(tree.score(X_val, y_val), 4),
          "auc", round(roc_auc_score(y_val, p), 4),
          "recall", round(recall_score(y_val, tree.predict(X_val)),4))

None | depth 23 leaves 848 | train 0.9979 val 0.7353 auc 0.6657 recall 0.5107
3 | depth 3 leaves 8 | train 0.7927 val 0.7906 auc 0.805 recall 0.3636
5 | depth 5 leaves 30 | train 0.8088 val 0.7885 auc 0.8137 recall 0.4973


- The unlimited tree does has train 0.9979 but val auc 0.6657 because just because it scores 0.9979 it does not mean it is good because The tree built 848 small groups for only 4,225 training rows, about 5 customers per leaf, so each leaf is nearly one customer’s memory. The name for this is overfitting, which it leads auc val score even worse 0.6657.
- val acc is 0.7353 barely above the baseline 0.7346, and memorising tree is useless on new customers because it memorized on specific customers, and if new employee enters, it will flag theh employee wether churn or not based on memorized.

In [31]:
def make_forest(**kw):
    params = dict(n_estimators=300, random_state=42, n_jobs=-1)
    params.update(kw)
    return Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(**params)),
    ])

In [32]:
rf = make_forest(oob_score=True).fit(X_tr, y_tr)
p_rf = rf.predict_proba(X_val)[:, 1]
print("RF300 train", round(rf.score(X_tr, y_tr), 4),
        "oob", round(rf.named_steps["model"].oob_score_, 4),
        "val", round(rf.score(X_val, y_val), 4),
        "auc", round(roc_auc_score(y_val, p_rf), 4))
report("RF val", y_val, p_rf)


RF300 train 0.9979 oob 0.7811 val 0.7743 auc 0.804
RF val t=0.5: tn=901 fp=134 fn=184 tp=190 | acc = 0.7743 prec = 0.5864 rec = 0.5080 f1 = 0.5444


- the forest auc 0.804 outperforms the single tree auc 0.6657 because when it comes to make a prediction forest aggregates the tree and vote based on majority. Each tree trains on a different bootstrap sample of the rows, and each split sees only a random subset of the features. That makes the trees disagree in different places, and the vote then cancels those mistakes.
- But the forest auc 0.804 is worse than logistic regression auc 0.8357 because forest memorising training data which leads to overfitting. For that we should control the depth that growing into micro tress by implementing max_depth. 
- The useful reading is that OOB (0.7811) and validation accuracy (0.7743) are close, only 0.0068 apart. So OOB is a trustworthy free estimate of performance on unseen rows. Both sit far below train 0.9979, which confirms the overfitting without needing the validation set.

In [47]:
def run(**kw):
        m = make_forest(**kw).fit(X_tr, y_tr)
        p = m.predict_proba(X_val)[:, 1]
        return (round(m.score(X_tr, y_tr), 4), round(m.score(X_val, y_val), 4),
                round(roc_auc_score(y_val, p), 4),
                round(recall_score(y_val, (p >= 0.5).astype(int)), 4))


In [48]:
print("n_estimators")
for k in (1, 10, 50, 100, 300, 600):  
    print(" ", k, run(n_estimators=k))
print("max_depth")
for d in (3, 5, 8, 12, 20, None):     
    print(" ", d, run(max_depth=d))
print("min_samples_split")
for m in (2, 10, 30, 100):            
    print(" ", m, run(min_samples_split=m))
print("min_samples_leaf")
for m in (1, 3, 5, 10, 20, 50):       
    print(" ", m, run(min_samples_leaf=m))


n_estimators
  1 (0.9004, 0.7076, 0.6376, 0.4866)
  10 (0.9796, 0.7686, 0.7788, 0.5535)
  50 (0.9972, 0.7644, 0.7977, 0.508)
  100 (0.9976, 0.7672, 0.7989, 0.4893)
  300 (0.9979, 0.7743, 0.804, 0.508)
  600 (0.9979, 0.7771, 0.8068, 0.5053)
max_depth
  3 (0.7903, 0.7842, 0.8342, 0.3021)
  5 (0.8036, 0.7984, 0.8366, 0.4225)
  8 (0.8376, 0.8034, 0.8345, 0.4947)
  12 (0.9444, 0.7793, 0.8179, 0.4893)
  20 (0.9979, 0.7708, 0.8041, 0.4973)
  None (0.9979, 0.7743, 0.804, 0.508)
min_samples_split
  2 (0.9979, 0.7743, 0.804, 0.508)
  10 (0.8914, 0.7871, 0.8239, 0.484)
  30 (0.8447, 0.7899, 0.8316, 0.4652)
  100 (0.8135, 0.7977, 0.8366, 0.4519)
min_samples_leaf
  1 (0.9979, 0.7743, 0.804, 0.508)
  3 (0.8836, 0.7899, 0.8247, 0.4893)
  5 (0.8573, 0.7913, 0.8303, 0.4786)
  10 (0.8312, 0.7913, 0.835, 0.4786)
  20 (0.8161, 0.7977, 0.8362, 0.4599)
  50 (0.8062, 0.797, 0.8365, 0.4198)


- n_estimators AUC goes 0.6376 -> 0.8068. When trees are added the validation AUC climbs up as majority vote stabelizes. It is not overfit, because it does not matter how many trees are adding to random forest, the training accuracy will stay locked at one point 0.9979 at 300 and 600 while the validation metrics will slowly improve or remain same. It is not worth since adding more trees does improve AUC by microscopic like 0.0028 while the memory usage and computational time increase.
- At depth = 5 the training accuracy stops memorizing and hit the highest validation AUC 0.8366. At depth = 12 the training accuracy hits 0.9444 which means the model started to build more branches which leads to drop the validation AUC from 0.8366 -> 0.8179 which is overfiting. At unlimited, the training accuracy hits the highest 0.9979 but hit the lowest validation AUC 0.804 which tells you that the model started to memorising training set.
- Min sample split at 100 (0.8366) gives the best validation AUC. As the trees restricted the recall score decrease.
- The three complexity limits (max_depth, min_samples_split, min_samples_leaf) made the biggest difference. Each lifted validation AUC from 0.804 to about 0.8366, a gain of 0.033. n_estimators made almost none beyond 50 trees (0.7977 → 0.8068).

- The key lesson of Part 5: more trees can’t fix trees that each memorise. Limiting how complex each tree is can. The forest only becomes competitive with logistic regression (0.8357) after you limit the trees.

In [51]:
rows = []
for d in (5, 8, 12, None):
    for leaf in (1, 5, 10, 20):
        tr_acc, val_acc, auc, rec = run(max_depth=d, min_samples_leaf=leaf)
        rows.append((str(d), leaf, tr_acc, val_acc, auc))
grid = pd.DataFrame(rows, columns=["max_depth", "min_samples_leaf",
                                    "train_acc", "val_acc", "val_auc"])
print(grid.sort_values("val_auc", ascending=False).head(5).to_string(index=False))

best = grid.loc[grid["val_auc"].idxmax()]
best_depth = None if best["max_depth"] == "None" else int(best["max_depth"])
best_leaf = int(best["min_samples_leaf"])
print("best", best_depth, best_leaf)


max_depth  min_samples_leaf  train_acc  val_acc  val_auc
        5                 5     0.8028   0.7999   0.8368
        8                10     0.8194   0.7999   0.8367
        5                10     0.8028   0.7963   0.8366
        5                 1     0.8036   0.7984   0.8366
        8                20     0.8159   0.8020   0.8365
best 5 5


- No because a microscopic difference of 0.0003 AUC does not mean 5 is better than 8 and 10. This minute change is simply noise. 
- Gap between train acc (0.8028) and val acc (0.7999) is 0.0029 compare to default forest between 0.9979 and 0.7743 is 0.2236 which tells you the the default forest memorise the training history which leads to collapsed in auc to 0.7743 when exposed to unseen data.

In [57]:
tuned = make_forest(max_depth=best_depth, min_samples_leaf=best_leaf).fit(X_tr, y_tr)
p_val = tuned.predict_proba(X_val)[:, 1]
print("tuned val auc", round(roc_auc_score(y_val, p_val), 4),
      "train", round(tuned.score(X_tr, y_tr), 4),
      "val", round(tuned.score(X_val, y_val), 4))

rows = []
for t in np.round(np.arange(0.20, 0.61, 0.05), 2):
    pred = (p_val >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, pred).ravel()
    rows.append((t, tn, fp, fn, tp, 
                 round(precision_score(y_val, pred), 4),
                 round(recall_score(y_val, pred), 4), 
                 round(f1_score(y_val, pred), 4)))
scan = pd.DataFrame(rows, columns=["t", "tn", "fp", "fn", "tp", "prec", "rec", "f1"])
print(scan.to_string(index=False))

ok = scan[scan["rec"] >= 0.70]
chosen_t = float(ok.loc[ok["f1"].idxmax(), "t"])
print("chosen threshold", chosen_t)

tuned val auc 0.8368 train 0.8028 val 0.7999
   t   tn  fp  fn  tp   prec    rec     f1
0.20  619 416  44 330 0.4424 0.8824 0.5893
0.25  701 334  70 304 0.4765 0.8128 0.6008
0.30  770 265  84 290 0.5225 0.7754 0.6243
0.35  810 225  99 275 0.5500 0.7353 0.6293
0.40  872 163 119 255 0.6100 0.6818 0.6439
0.45  923 112 156 218 0.6606 0.5829 0.6193
0.50  967  68 214 160 0.7018 0.4278 0.5316
0.55  990  45 258 116 0.7205 0.3102 0.4336
0.60 1007  28 288  86 0.7544 0.2299 0.3525
chosen threshold 0.35


- The default 0.5 is poor choice for this business because it values avoiding false alarms which leads to miss to catch the churners.
- At threshold 0.35 we would contact 500 customers. We reach 275 of the 374 who would leave (74%), but 225 of the 500 we contact (45%) were going to stay anyway.

In [ ]:
final = make_forest(max_depth=best_depth, min_samples_leaf=best_leaf).fit(X_train, y_train)
p_test = final.predict_proba(X_test)[:, 1]
print("Test auc", round(roc_auc_score(y_test, p_test), 4), 
      "Train Auc", round(final.score(X_train, y_train), 4), 
      "Test Auc", round(final.score(X_test, y_test), 4))
report("TEST RF", y_test, p_test, 0.5)
report("TEST RF", y_test, p_test, chosen_t)

lr_final = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
]).fit(X_train, y_train)

p_lr_test = lr_final.predict_proba(X_test)[:, 1]
print("TEST LR auc", round(roc_auc_score(y_test, p_lr_test), 4))
report("TEST LR", y_test, p_lr_test, chosen_t)

base_f = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("Baseline test acc", round(accuracy_score(y_test, base_f.predict(X_test)), 4))
default_f = make_forest().fit(X_train, y_train)
print("Default forest TEST auc", 
      round(roc_auc_score(y_test, default_f.predict_proba(X_test)[:, 1]), 4))

Test auc 0.8404 Train Auc 0.8026 Test Auc 0.7928
TEST RF t=0.5: tn=954 fp=81 fn=211 tp=163 | acc = 0.7928 prec = 0.6680 rec = 0.4358 f1 = 0.5275
TEST RF t=0.35: tn=814 fp=221 fn=105 tp=269 | acc = 0.7686 prec = 0.5490 rec = 0.7193 f1 = 0.6227
TEST LR auc 0.8383
TEST LR t=0.35: tn=810 fp=225 fn=105 tp=269 | acc = 0.7658 prec = 0.5445 rec = 0.7193 f1 = 0.6198
Baseline test acc 0.7346
Default forest TTEST auc 0.8183



Model                           Test AUC	                Test Recall @ 0.35
Baseline	            — (Accuracy Floor: 0.7346)	                  —
Logistic Regression (LR)	        0.8383	                        0.7193
Default Forest	                    0.8183	                          —
Tuned Forest ("5 and 5")	        0.8404	                        0.7193

- No, the forest cannot be declared decisively better. A tiny gap of 0.0021 AUC is pure statistical noise.
- The lesson is, if the default forest left untuned, the trees will grow deeper and leads to memorize background noise and overfit for training set.
- No, because a tiny drop of less than 1% between your training and testing sets is a textbook sign of a perfectly generalized, highly stable model.


In [59]:
imp_final = pd.Series(final.named_steps["model"].feature_importances_,
                         index=X.columns).sort_values(ascending=False)
print(imp_final.head(8).round(3).to_dict())

imp_default = pd.Series(default_f.named_steps["model"].feature_importances_,
                           index=X.columns).sort_values(ascending=False)
print(imp_default.head(5).round(3).to_dict())

perm = permutation_importance(tuned, X_val, y_val, scoring="roc_auc", n_repeats=10,
                              random_state=42, n_jobs=-1)
perm_s = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
print(perm_s.head(6).round(4).to_dict())

{'Contract_Month-to-month': 0.199, 'tenure': 0.147, 'TechSupport_No': 0.105, 'TotalCharges': 0.093, 'InternetService_Fiber optic': 0.091, 'PaymentMethod_Electronic check': 0.077, 'Contract_Two year': 0.06, 'MonthlyCharges': 0.057}
{'TotalCharges': 0.241, 'MonthlyCharges': 0.222, 'tenure': 0.183, 'Contract_Month-to-month': 0.059, 'TechSupport_No': 0.037}
{'Contract_Month-to-month': 0.0215, 'tenure': 0.0194, 'InternetService_Fiber optic': 0.0185, 'TotalCharges': 0.0081, 'TechSupport_No': 0.0062, 'Contract_Two year': 0.0055}


- The Top 3 Features: Contract_Month-to-month (0.199), tenure (0.147), and TechSupport_No (0.105). Yes, they align perfectly because day 9 Exploratory Data Analysis (EDA) highlighted that month-to-month users have a very high churn rate of 0.4271 and that leavers drop out early with a low average tenure of 17.98 months. The tuned forest correctly isolated these exact behavioral insights as its primary architectural split points, confirming that the model's intelligence mirrors the real economic data.
- Because TotalCharges and MonthlyCharges contain thousands of unique floating-point variations, an unlimited, overfit tree can split on them hundreds of times to memorize tiny noise patterns or specific row anomalies.
- No, feature_importances_ does not tell you the direction of the relationship. Random Forest feature importance scores are strictly absolute fractions between 0.0 and 1.0. They only tell you how much raw structural influence a column had on the model's splitting process.

## Conclusion
- I used the same churn data and the same splits as Day 9, so the
comparison is fair. The baseline (always "stayed") scores 0.7346
accuracy, and the Day 9 logistic regression reached a validation AUC
of 0.8357, which was the number to beat. A single unlimited decision
tree memorised the training data: 848 leaves, train accuracy 0.9979
but validation AUC only 0.6657 and validation accuracy 0.7353, barely
above the baseline. The default Random Forest (300 trees) was much
better than one tree (validation AUC 0.804) but still worse than
logistic regression, because each tree still memorised (train accuracy
0.9979). In the experiments, adding trees helped little beyond 50 (AUC
0.7977 to 0.8068 at 600), while limiting each tree mattered most:
max_depth=5, min_samples_split=100 and min_samples_leaf=50 all lifted
validation AUC from 0.804 to about 0.8366. I chose max_depth=5 and
min_samples_leaf=5 on validation AUC (0.8368), where train accuracy
0.8028 and validation accuracy 0.7999 are almost equal, but the top
five combinations differed by only 0.0003, which is noise. Choosing the
threshold with the same rule as Day 9 (recall of at least 0.70, then
best F1) gave 0.35. On the test set, used once, the tuned forest had
AUC 0.8404 against 0.8383 for logistic regression, and both caught 269
of the 374 leavers (recall 0.7193), so they are effectively tied; the
untuned default forest was clearly worse (AUC 0.8183). The most
important features were month-to-month contract (0.199), tenure (0.147)
and no tech support (0.105), and permutation importance on validation
agreed on contract and tenure; importance shows what the model used,
not what causes churn. Caveats: the test set has only 374 leavers, one
split is noisy, and a forest needs its settings checked on validation
data. Next step: cross-validation, to average over several splits
instead of trusting one.